In [8]:
import re, string

import numpy as np
import polars as pl

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

data = pl.read_csv("../../data/train.csv")

In [9]:
# Calculate the frequency distribution of the correct answer (A, B, C, D, E) 
# in train.csv. Based on your counts, what is the sum of the occurrences of 
# the most frequent option, and the least frequent option?  
 
ans_counts = data["answer"].value_counts()

max_freq = ans_counts["count"].max()
min_freq = ans_counts["count"].min()
q1_sum = max_freq + min_freq

print(f"Sum of most and least frequent option occurrences: {q1_sum}\n")

Sum of most and least frequent option occurrences: 814



In [10]:
# After converting the prompt column to lowercase, and removing all standard 
# punctuation characters (using Python's string.punctuation), split the text by 
# whitespace. What is the total number of unique words (vocabulary size) 
# across the entire cleaned prompt column of train.csv?  

punct_pattern = f"[{re.escape(string.punctuation)}]"

data = data.with_columns(
    pl.col("prompt")
    .str.to_lowercase()
    .str.replace_all(punct_pattern, "")
    .alias("cleaned_prompt")
)

unique_words_count = (
    data["cleaned_prompt"]
    .str.extract_all(r"\S+")
    .explode()
    .drop_nulls()
    .n_unique()
)

print(f"Total number of unique words: {unique_words_count}\n")


Total number of unique words: 859



In [11]:
# Using the cleaned prompt from Row ID 1, filter out the standard English stop words 
# using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words 
# are left in the prompt for Row ID 1 after filtering?  

row1_cleaned_prompt = data["cleaned_prompt"][0]

row1_words = row1_cleaned_prompt.split()

filtered_words = [word for word in row1_words if word not in ENGLISH_STOP_WORDS]

print(f"Words left in Row 1 prompt after filtering stop words: {len(filtered_words)}\n")

Words left in Row 1 prompt after filtering stop words: 13



In [12]:
# Fit a default TfidfVectorizer(stop_words='english') on a list containing all the 
# combined text of the prompts and options in train.csv. What is the exact total 
# number of feature columns (vocabulary size) generated by the vectorizer?  

text_columns = ["prompt", "A", "B", "C", "D", "E"]

all_text = []
for col in text_columns:
    all_text.extend(data[col].drop_nulls().to_list())

vectorizer = TfidfVectorizer(stop_words="english")
vectorizer.fit(all_text)

vocab_size = len(vectorizer.vocabulary_)

print(f"Total number of feature columns (vocabulary size): {vocab_size}\n")

Total number of feature columns (vocabulary size): 2762



In [13]:
# Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity 
# between the prompt and option A strictly for Row ID 1. What is the resulting 
# similarity score? (Round to 4 decimal places).

# Note: The question asks to use the vectorizer "fitted in Question 3", but it was fitted in Question 4.

prompt_row1 = data["prompt"][0]
option_A_row1 = data["A"][0]

tfidata_matrix_q5 = vectorizer.transform([prompt_row1, option_A_row1])

sim_score = cosine_similarity(tfidata_matrix_q5[0:1], tfidata_matrix_q5[1:2])[0][0]

print(f"Cosine similarity between prompt and Option A (Row 1): {sim_score:.4f}\n")

Cosine similarity between prompt and Option A (Row 1): 0.2328



In [ ]:
# Expand the logic from Question 4: For every row in train.csv, calculate the 
# cosine similarity between the prompt and each of its 5 options. 
# Then calculate the percentage of instances where the option with
# the highest cosine similarity matches the correct answer.   

prompt_tfidf = vectorizer.transform(data["prompt"].to_list())
A_tfidf = vectorizer.transform(data["A"].to_list())
B_tfidf = vectorizer.transform(data["B"].to_list())
C_tfidf = vectorizer.transform(data["C"].to_list())
D_tfidf = vectorizer.transform(data["D"].to_list())
E_tfidf = vectorizer.transform(data["E"].to_list())

def rowwise_cosine_sim(mat1, mat2):
    return np.array(mat1.multiply(mat2).sum(axis=1)).flatten()

sim_A = rowwise_cosine_sim(prompt_tfidf, A_tfidf)
sim_B = rowwise_cosine_sim(prompt_tfidf, B_tfidf)
sim_C = rowwise_cosine_sim(prompt_tfidf, C_tfidf)
sim_D = rowwise_cosine_sim(prompt_tfidf, D_tfidf)
sim_E = rowwise_cosine_sim(prompt_tfidf, E_tfidf)

all_sims = np.vstack([sim_A, sim_B, sim_C, sim_D, sim_E]).T

best_idx = np.argmax(all_sims, axis=1)
options_array = np.array(["A", "B", "C", "D", "E"])
predicted_answers = options_array[best_idx]

true_answers = np.array(data["answer"].to_list())
accuracy = (predicted_answers == true_answers).mean() * 100

print(f"Percentage where highest similarity matches correct answer: {accuracy:.2f}%")

Percentage where highest similarity matches correct answer: 13.70%


If the ground truth answer for a question is C, 
what is the MAP@3 score if a model predicts C A B?  

Since the ground truth is in the first position of the prediction, the correct answer is $$/frac{1/1} = 1$$